In [ ]:
import os
import subprocess
import pandas as pd

In [ ]:

   
def clean_tmpqcxms(msrun_dir):
    """Remove all files directly inside TMPQCXMS, leaving TMP.X subdirs untouched."""
    tmpqcxms = os.path.join(msrun_dir, "TMPQCXMS")
    if not os.path.isdir(tmpqcxms):
        return
    for item in os.listdir(tmpqcxms):
        item_path = os.path.join(tmpqcxms, item)
        if os.path.isfile(item_path):
            os.remove(item_path)
            print(f"  Removed: {item_path}")

WRKDIR    = "../data/simulation_results/franklin_tms/0039_10_ps_iee_05"
PLOTMS_BIN = os.path.expanduser("~/PlotMS.v.6.2.0/plotms")
GETRES_BIN = os.path.expanduser("getres")  # update this path

IEE_DIRS = [f"0039_iee{tag}" for tag in ["01", "02", "03", "04", "05", "06", "07", "08"]]

for iee_dir in IEE_DIRS:
    msrun_dir = os.path.join(WRKDIR, iee_dir, "GS-opt", "MS-run")

    if not os.path.isdir(msrun_dir):
        print(f"[SKIP] MS-run not found: {iee_dir}")
        continue

    # Check QCxMS finished normally
    qcxms_out = os.path.join(msrun_dir, "qcxms.out")
    if not os.path.exists(qcxms_out):
        print(f"[SKIP] No qcxms.out in {iee_dir}")
        continue
    with open(qcxms_out) as f:
        if "normal termination of QCxMS" not in f.read():
            print(f"[SKIP] QCxMS did not terminate normally in {iee_dir}")
            continue

    # Run getres
    clean_tmpqcxms(msrun_dir)  # clean TMPQCXMS files before processing
    print(f"Running getres for {iee_dir}...")
    try:
        subprocess.run(
            [GETRES_BIN],
            cwd=msrun_dir,
            check=True
        )
        print(f"  getres done: {iee_dir}")
    except subprocess.CalledProcessError as e:
        print(f"  [ERROR] getres failed for {iee_dir}: {e}")
        continue
    # Run PlotMS
    output_file = os.path.join(msrun_dir, "plotms.res")
    print(f"Running PlotMS for {iee_dir}...")
    try:
        with open(output_file, "w") as out_f:
            subprocess.run(
                [PLOTMS_BIN],
                cwd=msrun_dir,
                stdout=out_f,
                stderr=subprocess.STDOUT,
                check=True
            )
        print(f"  PlotMS done: {iee_dir} → {output_file}")
    except subprocess.CalledProcessError as e:
        print(f"  [ERROR] PlotMS failed for {iee_dir}: {e}")

  


In [ ]:
import os
import subprocess

# -----------------------------
# Configuration
# -----------------------------
WRKDIR = "../data/simulation_results/franklin_tms/0039_10_ps_iee_05"
script_path = os.path.abspath(os.path.expanduser(
    "../src/processing/bin_scale_intensities.py"  # update this path
))
OVERRIDE = False

IEE_DIRS       = [f"0039_iee{tag}" for tag in ["01", "02", "03", "04", "05", "06", "07", "08"]]
EXPECTED_SPECTRA = ["spectra_all.csv", "spectra_10pct.csv", "spectra_top20.csv"]


import glob


# -----------------------------
# Helpers
# -----------------------------
def spectra_complete(spectra_dir):
    return all(os.path.exists(os.path.join(spectra_dir, f)) for f in EXPECTED_SPECTRA)

def process(label, input_file, output_prefix):
    result = subprocess.run(
        ["python", script_path, "-i", input_file, "-o", output_prefix],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  [{label}] ERROR:\n{result.stderr}")
    else:
        print(f"  [{label}] Done")

# -----------------------------
# Main loop
# -----------------------------
for iee_dir in IEE_DIRS:
    msrun_dir = os.path.join(WRKDIR, iee_dir, "GS-opt", "MS-run")

    if not os.path.isdir(msrun_dir):
        print(f"[SKIP] MS-run not found: {iee_dir}")
        continue

    input_file = os.path.join(msrun_dir, "result.csv")
    if not os.path.exists(input_file):
        print(f"[SKIP] result.csv not found: {iee_dir}")
        continue

    spectra_dir = os.path.join(msrun_dir, "spectra")

    if not OVERRIDE and spectra_complete(spectra_dir):
        print(f"[SKIP] Already processed: {iee_dir}")
        continue

    os.makedirs(spectra_dir, exist_ok=True)

    process(iee_dir, input_file, os.path.join(spectra_dir, "spectra"))

In [ ]:
import os
import subprocess
import shutil

# -----------------------------
# Configuration
# -----------------------------
WRKDIR      = "../data/simulation_results/franklin_tms/0039_10_ps_iee_05"
script_path = os.path.abspath("../src/analysis/compare_spectra.py")
IEE_DIRS    = [f"0039_iee{tag}" for tag in ["01", "02", "03", "04", "05", "06", "07", "08"]]
MOL         = "0039"

# -----------------------------
# Sync EXP from parent franklin_tms (only MOL)
# -----------------------------
def sync_exp(mol=MOL):
    src = os.path.abspath(os.path.join(WRKDIR, "..", "EXP", mol))
    dst = os.path.join(WRKDIR, "EXP", mol)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copytree(src, dst)
    print(f"EXP synced: {src} -> {dst}")

# -----------------------------
# Link spectra into method/mol/spectra structure
# -----------------------------
def link_spectra_as_method(iee_dir, mol=MOL):
    src = os.path.join(WRKDIR, iee_dir, "GS-opt", "MS-run", "spectra")
    dst = os.path.join(WRKDIR, iee_dir, mol, "spectra")
    if not os.path.isdir(src):
        print(f"  [SKIP] No spectra in MS-run: {iee_dir}")
        return False
    if os.path.exists(dst):
        shutil.rmtree(dst)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copytree(src, dst)
    print(f"  Linked: {dst}")
    return True

# -----------------------------
# Step 1: sync EXP
# -----------------------------
sync_exp()

# -----------------------------
# Step 2: link all iee spectra
# -----------------------------
linked = []
for iee_dir in IEE_DIRS:
    if link_spectra_as_method(iee_dir):
        linked.append(iee_dir)

if not linked:
    print("No methods linked — aborting.")
else:
    # -----------------------------
    # Step 3: run compare once with all methods
    # -----------------------------
    shutil.rmtree(os.path.join(WRKDIR, "results"), ignore_errors=True)

    cmd = [
        "python", script_path,
        "--base_dir",      WRKDIR,
        "--methods",       *linked,
        "--include_all_peaks",
        "--include_top20",
        "--include_top10",
    ]
    print(f"\nRunning compare_spectra.py with methods: {linked}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[ERROR]:\n{result.stderr}")
    else:
        print(result.stdout)
        print("Done.")

In [ ]:
# Loadimport pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap

# -----------------------------
# Shared config
# -----------------------------
PALETTE = {
    "QCxMS":  "#D36EA5",   # was QCxMS_25_ps
    "QCxMS2":  "#4C4C4C",
    "CFMID":  "#A6A6A6",
    "NEIMS": "#7B68EE"
}
PALETTE = {
    "QCxMS": "#5A99D3",
    "QCxMS2":       "#4C4C4C",
    "CFMID":       "#A6A6A6",
    "NEIMS":      "#D36EA5"
}

METRICS = ["Cosine", "Weighted_Dot", "Tanimoto", "%S_sim_in_ref", "%R_ref_in_sim"]

METRIC_LABELS = {
    "Cosine":        "Cosine",
    "Weighted_Dot":  "Weighted Dot",
    "Tanimoto":      "Tanimoto",
    "%S_sim_in_ref": "Precision",
    "%R_ref_in_sim": "Recall"
}

PEAK_LABELS = {
    "all":   "All",
    "top20": "Top 20",
    "10pct": "≥10%"
}

PEAK_ORDER = ["all", "top20", "10pct"]

CMAP = LinearSegmentedColormap.from_list(
    "custom", ["#D4EFFF", "#A8D8EA", "#7BBDD6", "#A8D8EA", "#5A99D3", "#2E5FA3",  "#D36EA5", "#8B2252"], N=256
)


MIN_MOLECULES = 20  # minimum molecules required to include a method

# -----------------------------
# Figure sizing
# A standard 16:9 slide is 13.33 × 7.5 inches.
# One third of that width = ~4.4 inches. Height kept compact at 2.8.
# All font sizes set explicitly to 24pt (labels) / 20pt (ticks).
# DPI=300 so the image is sharp when placed on a slide.
# -----------------------------
FIG_W = 4.4
FIG_H = 5.5
DPI   = 300


# -----------------------------
# 1. Load data
# -----------------------------
def load_results(base_dir):
    """Load all comparison CSVs from results folder into a single DataFrame.
    QCxMS_25_ps is kept and renamed to QCxMS; QCxMS_10_ps is dropped.
    """
    results_dir = Path(base_dir) / "results"
    files = list(results_dir.glob("*/*.csv"))
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["Molecule"]  = f.parent.name
        df["Peak_Type"] = f.stem.split("_")[1]
        dfs.append(df)
    data = pd.concat(dfs, ignore_index=True)

    # Drop 10 ps, rename 25 ps → QCxMS
    data = data[data["Method"] != "QCxMS_10_ps"]
    data["Method"] = data["Method"].replace({"QCxMS_25_ps": "QCxMS"})

    print(f"Loaded {data.shape[0]} rows from {len(files)} files — {base_dir}")
    return data


def get_active_methods(data):
    """
    Return methods that have data for at least MIN_MOLECULES molecules.
    Filters dynamically from data, ordered by PALETTE if present.
    """
    mol_counts = (
        data.dropna(subset=METRICS, how="all")
        .groupby("Method")["Molecule"]
        .nunique()
    )
    active = mol_counts[mol_counts >= MIN_MOLECULES].index.tolist()

    # preserve palette order where possible, append any unknown methods at end
    ordered = [m for m in PALETTE if m in active]
    ordered += [m for m in active if m not in ordered]

    excluded = mol_counts[mol_counts < MIN_MOLECULES]
    if not excluded.empty:
        print("Methods excluded (insufficient data):")
        for m, n in excluded.items():
            print(f"  {m}: {n} molecules")

    print(f"Active methods: {ordered}")
    return ordered


# -----------------------------
# 2. Histogram plots
# -----------------------------
def plot_histograms(data, output_dir=None):
    if output_dir:
        Path(output_dir).mkdir(parents=True, exist_ok=True)

    method_order = get_active_methods(data)
    palette_spaced = {m.replace("_", " "): v for m, v in PALETTE.items() if m in method_order}
    method_order_spaced = [m.replace("_", " ") for m in method_order]

    plot_data = data.copy()
    plot_data["Method"] = plot_data["Method"].str.replace("_", " ")

    sns.set_style("white")
    sns.set_context("paper")   # neutral base; all sizes set explicitly below

    n_methods = len(method_order_spaced)
    BINS = 15

    for pt in sorted(data["Peak_Type"].unique()):
        subset = plot_data[
            (plot_data["Peak_Type"] == pt) &
            (plot_data["Method"].isin(method_order_spaced))
        ].dropna(subset=METRICS)
        pt_label = PEAK_LABELS.get(pt, pt)

        for metric in METRICS:
            metric_label = METRIC_LABELS[metric]
            fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
            fig.subplots_adjust(top=0.78)

            all_vals = subset[metric].dropna()
            bin_edges = np.linspace(all_vals.min(), all_vals.max(), BINS + 1)
            bin_width = bin_edges[1] - bin_edges[0]
            bar_width = bin_width / n_methods * 0.9

            for i, method in enumerate(method_order_spaced):
                vals = subset[subset["Method"] == method][metric].dropna()
                if vals.empty:
                    continue
                counts, _ = np.histogram(vals, bins=bin_edges, density=True)
                offsets = bin_edges[:-1] + i * bar_width
                ax.bar(offsets, counts, width=bar_width,
                       color=palette_spaced[method], alpha=0.85, label=method)

            ax.set_title(f"{metric_label} ({pt_label})", fontsize=24, weight="bold", pad=8)
            ax.set_xlabel(metric_label, fontsize=24)
            ax.set_ylabel("Density", fontsize=24)
            ax.tick_params(labelsize=20)
            leg = ax.legend(title="Method", fontsize=16, title_fontsize=18,
                            frameon=False, ncol=2,
                            loc='lower center', bbox_to_anchor=(0.5, 1.01))
            sns.despine(trim=False, ax=ax)
            plt.tight_layout()

            if output_dir:
                filename = Path(output_dir) / f"{metric_label} {pt_label}.png"
                plt.savefig(filename, dpi=DPI, bbox_inches="tight",
                            bbox_extra_artists=(leg,))
                print(f"Saved: {filename}")

            plt.show()
            plt.close()


# -----------------------------
# 3. Summary tables
# -----------------------------
def print_summary_tables(data):
    """Print mean ± std tables per peak type, only for active methods."""
    method_order = get_active_methods(data)

    for pt in sorted(data["Peak_Type"].unique()):
        subset = data[data["Peak_Type"] == pt].dropna(subset=METRICS)
        rows = []
        for method in method_order:
            method_data = subset[subset["Method"] == method]
            row = {"Method": method}
            for metric in METRICS:
                vals = method_data[metric].dropna()
                row[metric] = f"{vals.mean():.1f} ± {vals.std():.1f}" if len(vals) > 0 else "N/A"
            rows.append(row)
        df_table = pd.DataFrame(rows).set_index("Method")
        df_table.columns = [METRIC_LABELS[m] for m in df_table.columns]
        print(f"\n=== Peak Type: {pt} ===")
        display(df_table)


# -----------------------------
# 4. Heatmaps
# -----------------------------
def plot_heatmaps(data, output_dir=None):
    """Heatmap of mean ± std per method × peak type for each metric."""
    if output_dir:
        Path(output_dir).mkdir(parents=True, exist_ok=True)

    method_order = get_active_methods(data)

    sns.set_style("white")
    sns.set_context("paper")

    for metric, metric_label in METRIC_LABELS.items():
        fig, ax = plt.subplots(figsize=(FIG_W * 1.4, FIG_H))

        mean_matrix  = pd.DataFrame(
            index=[m.replace("_", " ") for m in method_order],
            columns=[PEAK_LABELS[p] for p in PEAK_ORDER]
        )
        annot_matrix = mean_matrix.copy()

        for method in method_order:
            for pt in PEAK_ORDER:
                vals = data[
                    (data["Method"] == method) &
                    (data["Peak_Type"] == pt)
                ][metric].dropna()
                mean_matrix.loc[method.replace("_", " "), PEAK_LABELS[pt]] = (
                    vals.mean() if len(vals) > 0 else np.nan)
                annot_matrix.loc[method.replace("_", " "), PEAK_LABELS[pt]] = (
                    f"{vals.mean():.1f}" if len(vals) > 0 else "N/A")

        mean_matrix = mean_matrix.astype(float)

        sns.heatmap(
            mean_matrix,
            annot=annot_matrix,
            fmt="",
            cmap=CMAP,
            linewidths=0.5,
            linecolor="white",
            ax=ax,
            cbar_kws={"label": metric_label},
            annot_kws={"size": 16, "color": "white"}
        )

        ax.set_title(metric_label, fontsize=24, weight="bold", pad=8)
        ax.set_xlabel("Peak Strategy", fontsize=24)
        ax.set_ylabel("Method", fontsize=24)
        ax.tick_params(labelsize=20)
        sns.despine(trim=False)

        # Fix colorbar font sizes
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=16)
        cbar.set_label(metric_label, fontsize=16)

        plt.tight_layout()

        if output_dir:
            filename = Path(output_dir) / f"Heatmap {metric_label}.png"
            plt.savefig(filename, dpi=DPI, bbox_inches="tight")
            print(f"Saved: {filename}")

        plt.show()
        plt.close()


# -----------------------------
# 5. Run all
# -----------------------------
def run_all(base_dir, output_dir=None):
    """Load data and generate all plots and tables for a given base directory."""
    data = load_results(base_dir)
    print_summary_tables(data)
    plot_histograms(data, output_dir=output_dir)
    plot_heatmaps(data, output_dir=output_dir)
    return data



data_iee = load_results(WRKDIR)

# Override palette and labels for iee grid
PALETTE_IEE = {
    "0039_iee01": "#D4EFFF",
    "0039_iee02": "#A8D8EA",
    "0039_iee03": "#7BBDD6",
    "0039_iee04": "#A8D8EA",
    "0039_iee05": "#5A99D3",
    "0039_iee06": "#2E5FA3",
    "0039_iee07": "#D36EA5",
    "0039_iee08": "#8B2252",
}

In [ ]:
# Temporarily override globals
import copy
_orig_palette = copy.copy(PALETTE)
_orig_min     = MIN_MOLECULES
PALETTE.clear()
PALETTE.update(PALETTE_IEE)
MIN_MOLECULES = 1  # only 1 molecule per method

print_summary_tables(data_iee)
plot_histograms(data_iee,  output_dir="plots/iee_grid")
plot_heatmaps(data_iee,    output_dir="plots/iee_grid")

# Restore
PALETTE.clear()
PALETTE.update(_orig_palette)
MIN_MOLECULES = _orig_min

In [ ]:
print(data_iee["Method"].unique())

In [ ]:
import os
for root, dirs, files in os.walk(os.path.join(WRKDIR, "results")):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

WRKDIR   = "../data/simulation_results/franklin_tms/0039_10_ps_iee_05"
IEE_TAGS = ["01","02","03","04", "05", "06", "07", "08"]
IEE_VALS = [0.1, 0.2, 0.3, 0.4,   0.5,   0.6,   0.7,   0.8]
MOL      = "0039"

METRICS = ["Cosine", "Weighted_Dot", "Tanimoto", "%S_sim_in_ref", "%R_ref_in_sim"]
METRIC_LABELS = {
    "Cosine":        "Cosine",
    "Weighted_Dot":  "Weighted Dot",
    "Tanimoto":      "Tanimoto",
    "%S_sim_in_ref": "Sim. peak coverage (%)",
    "%R_ref_in_sim": "Exp. peak coverage (%)"
}
PEAK_ORDER  = ["all", "top20", "10pct"]
PEAK_COLORS = {"all": "#5A99D3", "top20": "#D36EA5", "10pct": "#4C4C4C"}
PEAK_LABELS = {"all": "All", "top20": "Top 20", "10pct": "≥10%"}

# ── Load ──────────────────────────────────────────────────────
rows = []
for tag, iee_val in zip(IEE_TAGS, IEE_VALS):
    iee_dir = f"0039_iee{tag}"
    for pt in PEAK_ORDER:
        f = Path(WRKDIR) / "results" / MOL / f"spectra_{pt}_comparison.csv"
        if not f.exists():
            print(f"Missing: {f}")
            continue
        df = pd.read_csv(f)
        df = df[df["Method"] == iee_dir].copy()
        df["Peak_Type"] = pt
        df["ieeatm"]    = iee_val
        rows.append(df)

data = pd.concat(rows, ignore_index=True)

# ── Plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(
    1, len(METRICS),
    figsize=(4 * len(METRICS), 4.5),
    facecolor="none"
)

for ax, metric in zip(axes, METRICS):
    for pt in PEAK_ORDER:
        subset = data[data["Peak_Type"] == pt].sort_values("ieeatm")
        ax.plot(
            subset["ieeatm"],
            subset[metric],
            color=PEAK_COLORS[pt],
            marker="o",
            linewidth=2,
            markersize=7,
            label=PEAK_LABELS[pt]
        )

    ax.set_title(METRIC_LABELS[metric], fontsize=14, weight="bold", pad=8)
    ax.set_xlabel("ieeatm (eV/atom)", fontsize=12)
    ax.set_xticks(IEE_VALS)
    ax.tick_params(labelsize=11)
    ax.set_facecolor("none")

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_ylabel("Score", fontsize=12)

# shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    title="Peak strategy",
    fontsize=11, title_fontsize=12,
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=3
)

plt.suptitle(f"Metric vs ieeatm — molecule {MOL}", fontsize=15, weight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"plots/iee_grid/ieeatm_trends_{MOL}.png", dpi=300,
            bbox_inches="tight", facecolor="none")
plt.show()
plt.close()

In [ ]:
# ── Load peak counts from spectra files ──────────────────────
peak_rows = []
for tag, iee_val in zip(IEE_TAGS, IEE_VALS):
    iee_dir = f"0039_iee{tag}"
    for pt in PEAK_ORDER:
        spectra_file = Path(WRKDIR) / iee_dir / "GS-opt" / "MS-run" / "spectra" / f"spectra_{pt}.csv"
        if not spectra_file.exists():
            continue
        n_peaks = pd.read_csv(spectra_file, header=None).shape[0]
        peak_rows.append({"ieeatm": iee_val, "Peak_Type": pt, "n_peaks": n_peaks})

peak_df = pd.DataFrame(peak_rows)

# ── Plot ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4), facecolor="none")
ax.set_facecolor("none")

for pt in PEAK_ORDER:
    subset = peak_df[peak_df["Peak_Type"] == pt].sort_values("ieeatm")
    ax.plot(subset["ieeatm"], subset["n_peaks"],
            color=PEAK_COLORS[pt], marker="o", linewidth=2,
            markersize=7, label=PEAK_LABELS[pt])

ax.set_xlabel("ieeatm (eV/atom)", fontsize=14)
ax.set_ylabel("Number of peaks", fontsize=14)
ax.set_title("Peak count vs ieeatm — 0039", fontsize=14, weight="bold")
ax.set_xticks(IEE_VALS)
ax.tick_params(labelsize=12)
ax.legend(frameon=False, fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("plots/iee_grid/peak_count_vs_ieeatm.png", dpi=300, bbox_inches="tight")
plt.show()